# 29b · External test set CDD-CESM (low-energy images, hand-drawn radiologist regions) — CPU, ~10 min
**Before running:** download CDD-CESM from TCIA (doi 10.7937/29kw-ae92; CC BY 4.0): the low-energy JPEG images, the annotation table and the
hand-drawn segmentation file, and put them (zip or folder) in `MyDrive/quantum-placement/data/external/cdd_cesm/`.
Rules (prespecified in the revision-5 protocol): low-energy (DM) images only; one lesion = one hand-drawn region typed as a mass; if the table
types findings per image only, regions of images whose findings include a mass and no other region-forming finding (calcification,
architectural distortion, asymmetry) are used; shapes rasterized as drawn; crops and window exactly as for CSAW-S (notebook 29).
Writes `data/external/cddcesm_mass_256_jit/` (images, masks, metadata.csv, inventory.json, sheet.png). No model is loaded here.

In [ ]:
# 1 · Settings and file discovery
from google.colab import drive
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
import os, glob, re, json, zipfile, hashlib
import numpy as np, pandas as pd, cv2
PROJECT = '/content/drive/MyDrive/quantum-placement'
SRC = f'{PROJECT}/data/external/cdd_cesm'; OUT = f'{PROJECT}/data/external/cddcesm_mass_256_jit'; LOCAL = '/content/cdd_cesm'
CROP_SIZE, SCALE_LO, SCALE_HI, SHIFT = 256, 1.5, 3.0, 0.8
os.makedirs(LOCAL, exist_ok=True)
for z in glob.glob(f'{SRC}/**/*.zip', recursive=True):
    print('unzip', os.path.basename(z)); zipfile.ZipFile(z).extractall(LOCAL)
for f in glob.glob(f'{SRC}/**/*', recursive=True):
    if os.path.isfile(f) and not f.endswith('.zip'):
        dst = os.path.join(LOCAL, os.path.relpath(f, SRC)); os.makedirs(os.path.dirname(dst), exist_ok=True)
        if not os.path.exists(dst): os.system(f'cp "{f}" "{dst}"')
imgs = {os.path.splitext(os.path.basename(p))[0]: p for p in glob.glob(f'{LOCAL}/**/*', recursive=True)
        if re.search(r'\.(jpe?g|png|tiff?)$', p, re.I)}
dm = {k: v for k, v in imgs.items() if re.search(r'_DM_', k)}
print('images:', len(imgs), '| low-energy (DM):', len(dm), '| example:', sorted(dm)[:3])
tabs = [p for p in glob.glob(f'{LOCAL}/**/*', recursive=True) if re.search(r'\.(csv|xlsx|xls)$', p, re.I)]
def read_any(p): return pd.read_excel(p) if p.lower().endswith(('xlsx', 'xls')) else pd.read_csv(p)
T = {p: read_any(p) for p in tabs}
for p, t in T.items(): print('\n', os.path.relpath(p, LOCAL), t.shape, '\n', list(t.columns)[:20])
SEG = [p for p, t in T.items() if any('region_shape_attributes' in str(c) for c in t.columns)]
ANN = [p for p, t in T.items() if p not in SEG and any('finding' in str(c).lower() for c in t.columns)]
assert len(dm) > 0 and len(SEG) == 1 and len(ANN) >= 1, f'files not found as expected: DM {len(dm)}, segmentation tables {SEG}, annotation tables {ANN}'
seg = T[SEG[0]]; ann = pd.concat([T[p] for p in ANN], ignore_index=True)
print('\nsegmentation file:', os.path.basename(SEG[0]), seg.shape, '| annotation:', [os.path.basename(p) for p in ANN], ann.shape)

In [ ]:
# 2 · Lesion table (rules of the protocol) and inventory
fcol = [c for c in ann.columns if 'finding' in str(c).lower()][0]
ncol = [c for c in ann.columns if re.search(r'image|file', str(c), re.I)][0]
pcol = ([c for c in ann.columns if 'patholog' in str(c).lower() or 'classif' in str(c).lower()] + [None])[0]
ann['_key'] = ann[ncol].astype(str).map(lambda s: os.path.splitext(os.path.basename(s.strip()))[0])
def kinds(text):
    t = str(text).lower().replace('non-mass', 'nonmass').replace('non mass', 'nonmass').replace('mass enhancement', 'massenh')
    return {'mass': bool(re.search(r'\bmass', t)), 'calc': 'calc' in t, 'distortion': 'distortion' in t, 'asym': 'asymmetr' in t}
fcol_seg = None
seg['_key'] = seg[[c for c in seg.columns if 'filename' in str(c).lower()][0]].astype(str).map(lambda s: os.path.splitext(os.path.basename(s))[0])
def ra(x):
    try: return json.loads(x) if isinstance(x, str) else {}
    except Exception: return {}
seg['_shape'] = seg['region_shape_attributes'].map(ra)
seg['_attr'] = seg['region_attributes'].map(ra) if 'region_attributes' in seg else [{}] * len(seg)
per_region = any(any(re.search(r'mass|calc|distortion|asym', str(v), re.I) for v in a.values()) for a in seg['_attr'])
print('finding type per region in the segmentation file:', per_region)
img_kind = ann.groupby('_key')[fcol].apply(lambda s: ' ; '.join(map(str, s))).map(kinds).to_dict()
rows, excl = [], {}
def X(reason): excl[reason] = excl.get(reason, 0) + 1
for i, r in seg.iterrows():
    k = r['_key']; sh = r['_shape']
    if k not in dm: X('not a low-energy (DM) image' if k in imgs else 'image file missing'); continue
    if not sh or sh.get('name') in (None, 'point', 'polyline'): X(f"no area shape ({sh.get('name') if sh else 'empty'})"); continue
    if per_region:
        kd = kinds(' '.join(map(str, r['_attr'].values())))
        if not kd['mass']: X('region not typed as mass'); continue
    else:
        kd = img_kind.get(k)
        if kd is None: X('image not in annotation table'); continue
        if not kd['mass']: X('image findings without mass'); continue
        if kd['calc'] or kd['distortion'] or kd['asym']: X('image with mass and another region-forming finding'); continue
    rows.append(dict(key=k, region=i, shape=sh))
L = pd.DataFrame(rows)
print('lesions:', len(L), '| images:', L.key.nunique() if len(L) else 0, '| exclusions:', json.dumps(excl, indent=1))

In [ ]:
# 3 · Masks, crops (study protocol), sheet
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
def crop_window(mask, rid, size=CROP_SIZE):
    ys, xs = np.where(mask > 0); y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1; Lm = max(y1 - y0, x1 - x0)
    rng = np.random.default_rng(int(hashlib.md5(rid.encode()).hexdigest()[:8], 16))
    scale = rng.uniform(SCALE_LO, SCALE_HI); side = max(int(Lm * scale), 64)
    sy, sx = (side - (y1 - y0)) / 2, (side - (x1 - x0)) / 2; dy, dx = rng.uniform(-SHIFT, SHIFT, 2)
    cy = int(round((y0 + y1) / 2 + dy * sy)); cx = int(round((x0 + x1) / 2 + dx * sx))
    return cy - side // 2, cx - side // 2, side, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side, scale=round(float(scale), 4),
                                                    shift_y=round(float(dy), 4), shift_x=round(float(dx), 4))
def cut(a, top, left, side, interp):
    H, W = a.shape; pad = max(0, -top, -left, top + side - H, left + side - W)
    if pad: a = np.pad(a, pad, mode='constant'); top += pad; left += pad
    return cv2.resize(a[top:top + side, left:left + side], (CROP_SIZE, CROP_SIZE), interpolation=interp), bool(pad)
def raster(sh, H, W):
    m = np.zeros((H, W), np.uint8); n = sh['name']
    if n == 'polygon':
        pts = np.stack([sh['all_points_x'], sh['all_points_y']], 1).astype(np.int32); cv2.fillPoly(m, [pts], 1)
    elif n == 'rect':
        cv2.rectangle(m, (int(sh['x']), int(sh['y'])), (int(sh['x'] + sh['width']), int(sh['y'] + sh['height'])), 1, -1)
    elif n == 'circle':
        cv2.circle(m, (int(sh['cx']), int(sh['cy'])), int(sh['r']), 1, -1)
    elif n == 'ellipse':
        cv2.ellipse(m, (int(sh['cx']), int(sh['cy'])), (int(sh['rx']), int(sh['ry'])), float(np.degrees(sh.get('theta', 0))), 0, 360, 1, -1)
    return m
for d in ('images', 'masks'): os.makedirs(f'{OUT}/{d}', exist_ok=True)
meta, small = [], 0
for k, g in L.groupby('key'):
    im = cv2.imread(dm[k], cv2.IMREAD_UNCHANGED)
    if im.ndim == 3: im = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    H, W = im.shape; g8 = to_uint8(im)
    for j, r in enumerate(g.itertuples()):
        les = raster(r.shape, H, W)
        if les.sum() == 0: small += 1; continue
        rid = f"cdd_{re.sub(r'[^A-Za-z0-9]', '', k)}_r{j}"
        top, left, side, info = crop_window(les, rid)
        ic, padded = cut(g8, top, left, side, cv2.INTER_AREA); mc, _ = cut(les, top, left, side, cv2.INTER_NEAREST)
        cv2.imwrite(f'{OUT}/images/{rid}.png', ic); cv2.imwrite(f'{OUT}/masks/{rid}.png', (mc > 0).astype(np.uint8) * 255)
        pat = k.split('_')[0]
        path = None
        if pcol is not None:
            pv = ann.loc[ann._key == k, pcol]; path = str(pv.iloc[0]) if len(pv) else None
        meta.append(dict(roi_id=rid, image=k, patient=pat, shape=r.shape['name'], pathology=path, padded=padded,
                         mask_frac=round(float((mc > 0).mean()), 4), lesion_frac_of_image=round(float(les.mean()), 5), **info))
M = pd.DataFrame(meta); M.to_csv(f'{OUT}/metadata.csv', index=False)
inv = dict(low_energy_images=len(dm), lesions=len(M), images_with_lesion=int(M.image.nunique()), patients=int(M.patient.nunique()),
           exclusions=excl, empty_rasters=small, per_region_typing=bool(per_region), shapes=M['shape'].value_counts().to_dict(),
           pathology=M.pathology.value_counts(dropna=False).to_dict())
json.dump(inv, open(f'{OUT}/inventory.json', 'w'), indent=2); print(json.dumps(inv, indent=1))
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
ids = M.roi_id.tolist()[:96]; cols = 8; rws = int(np.ceil(len(ids) / cols))
fig, ax = plt.subplots(rws, cols, figsize=(16, 2.2 * rws), squeeze=False)
for a_, r in zip(ax.ravel(), ids):
    im = cv2.cvtColor(cv2.imread(f'{OUT}/images/{r}.png', 0), cv2.COLOR_GRAY2RGB)
    cs, _ = cv2.findContours((cv2.imread(f'{OUT}/masks/{r}.png', 0) > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(im, cs, -1, (60, 220, 60), 2); a_.imshow(im); a_.set_title(r[-14:], fontsize=6)
for a_ in ax.ravel(): a_.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/sheet.png', dpi=80); print('DONE')